[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HSF-reco-and-software-triggers/Tracking-ML-Exa.TrkX/blob/master/Examples/TrackML_Quickstart/DM_colab_quickstart.ipynb)

# TrackML Quickstart

## Install Libraries

**Note: Before running notebook, ensure your runtime is set to GPU**

First, we just install a few libraries (this should take around 5 minutes and automatically restart the kernel), and load in the repository.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!pip install seaborn bokeh 
!conda install pandas scipy matplotlib cupy "cudatoolkit>=11.3" "pytorch>=1.10.2" "pytorch-lightning>=1.6" pyg faiss-gpu -c pytorch -c pyg -c conda-forge

In [ ]:
!git clone https://github.com/HSF-reco-and-software-triggers/Tracking-ML-Exa.TrkX.git
%cd Tracking-ML-Exa.TrkX/Examples/TrackML_Quickstart

# Import libraries

In [5]:
import sys, os
sys.path.append("../../")
from Scripts import train_metric_learning, run_metric_learning_inference, train_gnn, run_gnn_inference, build_track_candidates, evaluate_candidates
from Scripts.utils.convenience_utils import get_example_data, plot_true_graph, get_training_metrics, plot_training_metrics, plot_neighbor_performance, plot_predicted_graph, plot_track_lengths, plot_edge_performance, plot_graph_sizes
import yaml

import warnings
warnings.filterwarnings("ignore")
CONFIG = 'pipeline_config_quirk.yaml'
# pipeline_config_quirk.yaml

## Download Data

In [ ]:
%%capture
!mkdir datasets
!wget https://portal.nersc.gov/cfs/m3443/dtmurnane/TrackML_Example/trackml_quickstart_dataset.tar.gz -O datasets/trackml_quickstart_dataset.tar.gz

In [ ]:
%%capture
!tar -xvf datasets/trackml_quickstart_dataset.tar.gz -C datasets;
!rm datasets/trackml_quickstart_dataset.tar.gz

## TrackML Dataset

The TrackML dataset contains simulated indepedent proton-proton collision events, each generating hundreds of particles, each of which hits cells and layers of the detector layers multiple times. The detector records the spatial coordinates and other auxillary information of these hits which, if properly connected, form tracks associated with the parent particle and the collision event from which it originates. The challenge and goal of this project is to associate each and every hit to one single track with optimal purity and efficiency, whose precise definition will be given later.

Each entry in the particles data frame contains a unique identifier of the particle (particle_id), its charge (q), its initial position or vertex $(v_x, v_y, v_z)$, its initial momentum in GeV/c $(p_x, p_y, p_z)$ and its associated number of detector hits. 

Many particles do not leave behind any detector hits and obviously cannot be associated to any track. This is called "detector inefficiency". They are among "uninterested particles" and will be mostly filtered out by a simple momentum cut.

### Training data
Let us take a look at the data before training. In this example pipeline, we have preprocessed the TrackML data into a more convenient form. We calculated directional information and summary statistics from the charge deposited in each spacepoints, and append them to its cyclidrical coordinates. Let us load an example data file and inspect the content.

In [6]:
with open(CONFIG, 'r') as f:
    configs = yaml.load(f, Loader=yaml.FullLoader)

In [7]:
example_data_df, example_data_pyg = get_example_data(configs)
example_data_df.head()

,0,1,2
0,0.031687,0.567258,0.012397
1,0.071077,0.566757,0.028057
2,0.072884,0.566715,0.028778
3,0.170848,0.565448,0.067714
4,0.172741,0.565432,0.068462


### Visualize tracks

A "true track" is defined as a set of sequential hits, all left by the same particle. Therefore a true edge is the edge formed by two sequential hits. Let's visualize a random set of 200 true tracks:

In [8]:
plot_true_graph_fig = plot_true_graph(example_data_pyg, num_tracks=200)

# 1. Train Metric Learning

## Train metric learning model

Finally we come to model training. By default, we train the MLP for 30 epochs, which takes approximately 15 minutes on an NVidia V100. Feel free to adjust the epoch number in pipeline_config.yml

In [9]:
metric_learning_trainer, metric_learning_model = train_metric_learning(CONFIG)

INFO:-------------------- Step 1: Running metric learning training --------------------
INFO:----------------------------- a) Initialising model -----------------------------
INFO:------------------------------ b) Running training ------------------------------
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params
---------------------------------------
0 | network | Sequential | 3.2 M 
---------------------------------------
3.2 M     Trainable params
0   

Epoch 49: 100%|██████████| 90/90 [00:18<00:00,  4.86it/s, loss=0.00733, v_num=2]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 90/90 [00:18<00:00,  4.84it/s, loss=0.00733, v_num=2]

INFO:-------------------------------- c) Saving model --------------------------------


## this below is Rafia's asked cell: 

""Show the event statistics alongside the plot: SM 3000, quirk 36, anti quirk 68, total 3104, quirk fraction about 3.35 percent."

In [28]:
# Pick most quirk-rich event from trainset
best_i, best_q = -1, -1
for i, ev in enumerate(metric_learning_model.trainset):
    if hasattr(ev, "source_label"):
        q = int((ev.source_label > 0).sum())
        if q > best_q:
            best_i, best_q = i, q

ev = metric_learning_model.trainset[best_i]
src = ev.source_label.cpu()
sm = int((src == 0).sum())
q = int((src == 1).sum())
aq = int((src == 2).sum())
tot = int(src.numel())
qfrac = 100.0 * (q + aq) / max(1, tot)

print(f"Event stats -> SM {sm}, quirk {q}, anti quirk {aq}, total {tot}, quirk fraction {qfrac:.2f}%")
plot_true_graph_fig = plot_true_graph(ev, num_tracks=200)


Event stats -> SM 1102, quirk 36, anti quirk 68, total 1206, quirk fraction 8.62%


Try running the update-desktop-database command. If you
don't have this command you should install the
desktop-file-utils package. This package is available from
http://freedesktop.org/wiki/Software/desktop-file-utils/
No applications found for mimetype: text/html
./usr/bin/xdg-open: 882: x-www-browser: not found
/usr/bin/xdg-open: 882: firefox: not found
/usr/bin/xdg-open: 882: iceweasel: not found
/usr/bin/xdg-open: 882: seamonkey: not found
/usr/bin/xdg-open: 882: mozilla: not found
/usr/bin/xdg-open: 882: epiphany: not found
/usr/bin/xdg-open: 882: konqueror: not found
/usr/bin/xdg-open: 882: chromium: not found
/usr/bin/xdg-open: 882: chromium-browser: not found
/usr/bin/xdg-open: 882: google-chrome: not found
/usr/bin/xdg-open: 882: www-browser: not found
/usr/bin/xdg-open: 882: links2: not found
/usr/bin/xdg-open: 882: elinks: not found
/usr/bin/xdg-open: 882: links: not found
/usr/bin/xdg-open: 882: lynx: not found
/usr/bin/xdg-open: 882: w3m: not found
xdg-open: no method avai

 ## plot a high-quirk event 
### According to Rafia's requirement

In [10]:
best_i, best_q = -1, -1
for i, ev in enumerate(metric_learning_model.trainset):
    if hasattr(ev, "source_label"):
        q = int((ev.source_label > 0).sum())
        if q > best_q:
            best_i, best_q = i, q

print("best trainset index:", best_i, "quirk_hits:", best_q)
plot_true_graph_fig = plot_true_graph(metric_learning_model.trainset[best_i], num_tracks=200)


best trainset index: 42 quirk_hits: 104


## Plot training metrics

We can examine how the training went. This is stored in a simple dataframe:

In [11]:
embedding_metrics = get_training_metrics(metric_learning_trainer)
embedding_metrics.head()

,epoch,train_loss,val_loss,eff,pur,current_lr
0,0,0.008549,0.005259,0.876745,0.098513,0.000125
1,1,0.008342,0.005021,0.874886,0.143036,0.000250
2,2,0.008279,0.004973,0.871542,0.150084,0.000375
3,3,0.008215,0.004985,0.868409,0.157548,0.000350
4,4,0.008157,0.004918,0.871210,0.171080,0.000625


In [12]:
embedding_training_figs = plot_training_metrics(embedding_metrics)

## Evaluate model performance on sample test data

Here we evaluate the model performace on one sample test data. We look at how the efficiency and purity change with the embedding radius.

In [13]:
neighbor_perf_figs = plot_neighbor_performance(metric_learning_model)

## Plot example truth and predicted graphs

In [14]:
predicted_graph_figs = plot_predicted_graph(metric_learning_model)

## Track lengths

In [15]:
track_length_figs = plot_track_lengths(metric_learning_model)

In [16]:
graph_sizes_fig = plot_graph_sizes(metric_learning_model)

100%|██████████| 80/80 [00:12<00:00,  6.21it/s]


# 2. Construct graphs from metric learning inference

This step performs model inference on the entire input datasets (train, validation and test), to obtain input graphs to the graph neural network. Optionally, we also clear the directory.

In [17]:
graph_builder = run_metric_learning_inference(CONFIG)

INFO:------------- Step 2: Constructing graphs from metric learning model -------------
INFO:---------------------------- a) Loading trained model ----------------------------
INFO:----------------------------- b) Running inferencing -----------------------------


Training finished, running inference to build graphs...


100%|██████████| 10/10 [00:01<00:00,  6.27it/s]


In [18]:
import os, gc, torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"Cleared CUDA cache on {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available")

print("Run this cell right before Step 3. If allocator behavior is unchanged, restart kernel once.")


Cleared CUDA cache on NVIDIA GeForce RTX 4090
Run this cell right before Step 3. If allocator behavior is unchanged, restart kernel once.


# 3. Train graph neural networks

We have a set of graphs constructed. We now train a GNN to classify edges as either "true" (belonging to the same track) or "false" (not belonging to the same track).

In [19]:
gnn_trainer, gnn_model = train_gnn(CONFIG)

INFO:-------------------------  Step 3: Running GNN training  -------------------------
INFO:----------------------------- a) Initialising model -----------------------------
INFO:------------------------------ b) Running training ------------------------------
Using 16bit None Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                   | Type       | Params
------------------------------------------------------
0 | node_encoder           | Sequential | 9.9 K 
1 | edge_encoder           | Sequential | 28.0 K
2 | edge_network           | Sequential | 37.2 K
3 | node_network           | Sequential | 37.2 K
4 | output_edge_classifier | Sequential | 37.5 K
------------------------------------------------------
149 K     Trainable params
0         Non-trainable params
149 K     Total params
0.300    

Epoch 29: 100%|██████████| 90/90 [00:04<00:00, 18.71it/s, loss=0.231, v_num=2]

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 90/90 [00:04<00:00, 18.63it/s, loss=0.231, v_num=2]


INFO:-------------------------------- c) Saving model --------------------------------


## Plot training metrics

In [20]:
gnn_metrics = get_training_metrics(gnn_trainer)
gnn_metrics.head()

,epoch,train_loss,val_loss,eff,pur,current_lr
0,0,0.580200,0.574998,0.962851,0.787412,0.0002
1,1,0.556659,0.537790,0.948725,0.803301,0.0004
2,2,0.554551,0.529269,0.956538,0.799306,0.0006
3,3,0.548872,0.523916,0.964132,0.791510,0.0008
4,4,0.551140,0.520035,0.968433,0.787157,0.0010


In [21]:
gnn_training_figs = plot_training_metrics(gnn_metrics)

## Evaluate model performance on sample test data

Here we evaluate the model performace on one sample test data. We look at how the efficiency and purity change with the embedding radius.

In [22]:
edge_perf_figs = plot_edge_performance(gnn_model)

# Step 4: GNN inference 

In [23]:
run_gnn_inference(CONFIG)

INFO:--------------------- Step 4: Scoring graph edges using GNN  ---------------------
INFO:---------------------------- a) Loading trained model ----------------------------
INFO:----------------------------- b) Running inferencing -----------------------------


Training finished, running inference to filter graphs...
Building train


100%|██████████| 80/80 [00:00<00:00, 201.63it/s]


Building val


100%|██████████| 10/10 [00:00<00:00, 183.56it/s]


Building test


100%|██████████| 10/10 [00:00<00:00, 203.42it/s]


# Step 5: Build track candidates from GNN

In [24]:
build_track_candidates(CONFIG)

INFO:-----------  Step 5: Building track candidates from the scored graph  -----------
INFO:---------------------------- a) Loading scored graphs ----------------------------
INFO:---------------------------- b) Labelling graph nodes ----------------------------
100%|██████████| 100/100 [00:00<00:00, 674.87it/s]


# Step 6: Evaluate track candidates

We can control the matching style in the pipeline config file. The following all require at least a majority of hits to match in each scheme (i.e. matching fraction = 50%).
A discussion of each matching style and some worked examples can be found in the [Documentation](https://hsf-reco-and-software-triggers.github.io/Tracking-ML-Exa.TrkX/performance/matching_definitions/).

ATLAS style matching is the default.

In [25]:
evaluated_events, reconstructed_particles, particles, matched_tracks, tracks = evaluate_candidates(CONFIG)

INFO:------------ Step 6: Evaluating the track reconstruction performance ------------
INFO:--------------------------- a) Loading labelled graphs ---------------------------
100%|██████████| 100/100 [00:00<00:00, 167.84it/s]
INFO:--------------------- b) Calculating the performance metrics ---------------------
INFO:Number of reconstructed particles: 24524
INFO:Number of particles: 26762
INFO:Number of matched tracks: 30662
INFO:Number of tracks: 30917
INFO:Number of duplicate reconstructed particles: 6128
INFO:Efficiency: 0.916
INFO:Fake rate: 0.008
INFO:Duplication rate: 0.250
INFO:------------------------------ c) Plotting results ------------------------------


In [ ]:
# Optional: save selected plots from this notebook
import os
from pathlib import Path

import matplotlib.pyplot as plt
from bokeh.io import export_png

SAVE_PLOTS = True  # Set True when you want to export
OUTPUT_DIR = Path("saved_plots")
MATPLOTLIB_DPI = 500
BOKEH_SCALE_FACTOR = 5

# Choose what to save
TO_SAVE = [
    "plot_true_graph",
    "embedding_training",
    "neighbor_performance",
    "predicted_graph",
    "track_lengths",
    "graph_sizes",
    "gnn_training",
    "edge_performance",
]

plot_registry = {
    "plot_true_graph": locals().get("plot_true_graph_fig"),
    "embedding_training": locals().get("embedding_training_figs"),
    "neighbor_performance": locals().get("neighbor_perf_figs"),
    "predicted_graph": locals().get("predicted_graph_figs"),
    "track_lengths": locals().get("track_length_figs"),
    "graph_sizes": locals().get("graph_sizes_fig"),
    "gnn_training": locals().get("gnn_training_figs"),
    "edge_performance": locals().get("edge_perf_figs"),
}

def _bokeh_figures(obj):
    if obj is None:
        return []
    if isinstance(obj, dict) and "figures" in obj:
        return obj["figures"]
    return [obj]

def _save_matplotlib(fig, outpath):
    fig.savefig(outpath, dpi=MATPLOTLIB_DPI, bbox_inches="tight")

def _save_bokeh(fig, outpath):
    # Requires selenium + browser driver for PNG export.
    export_png(fig, filename=str(outpath), scale_factor=BOKEH_SCALE_FACTOR)

if SAVE_PLOTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    saved, skipped, failed = [], [], []

    for name in TO_SAVE:
        obj = plot_registry.get(name)
        if obj is None:
            skipped.append((name, "plot object not found (run the source plotting cell first)"))
            continue

        if name == "graph_sizes":
            try:
                out = OUTPUT_DIR / f"{name}.png"
                _save_matplotlib(obj, out)
                saved.append(str(out))
            except Exception as exc:
                failed.append((name, str(exc)))
            continue

        figs = _bokeh_figures(obj)
        for idx, fig in enumerate(figs, start=1):
            suffix = f"_{idx}" if len(figs) > 1 else ""
            out = OUTPUT_DIR / f"{name}{suffix}.png"
            try:
                _save_bokeh(fig, out)
                saved.append(str(out))
            except Exception as exc:
                failed.append((f"{name}{suffix}", str(exc)))

    print(f"Saved {len(saved)} plot files")
    if saved:
        print("\n".join(saved))
    if skipped:
        print("\nSkipped:")
        for n, msg in skipped:
            print(f"- {n}: {msg}")
    if failed:
        print("\nFailed:")
        for n, msg in failed:
            print(f"- {n}: {msg}")
else:
    print("Set SAVE_PLOTS = True and rerun this cell to export selected plots.")

In [ ]:
print(plot_true_graph_fig if "plot_true_graph_fig" in locals() else "missing")
print(embedding_training_figs if "embedding_training_figs" in locals() else "missing")


In [26]:
# Save notebook plots: Bokeh as individual HTML files, Matplotlib as PNG
from pathlib import Path

from bokeh.io import output_file, save
import matplotlib.pyplot as plt

SAVE_PLOTS = True
OUTPUT_DIR = Path("saved_plots")
MATPLOTLIB_DPI = 600

TO_SAVE = [
    "plot_true_graph",
    "embedding_training",
    "neighbor_performance",
    "predicted_graph",
    "track_lengths",
    "graph_sizes",
    "gnn_training",
    "edge_performance",
]

plot_registry = {
    "plot_true_graph": locals().get("plot_true_graph_fig"),
    "embedding_training": locals().get("embedding_training_figs"),
    "neighbor_performance": locals().get("neighbor_perf_figs"),
    "predicted_graph": locals().get("predicted_graph_figs"),
    "track_lengths": locals().get("track_length_figs"),
    "graph_sizes": locals().get("graph_sizes_fig"),
    "gnn_training": locals().get("gnn_training_figs"),
    "edge_performance": locals().get("edge_perf_figs"),
}

def _bokeh_figures(obj):
    if obj is None:
        return []
    if isinstance(obj, dict) and "figures" in obj:
        return obj["figures"]
    return [obj]

def _save_matplotlib(fig, outpath):
    fig.savefig(outpath, dpi=MATPLOTLIB_DPI, bbox_inches="tight")

def _save_bokeh_html(fig, outpath):
    output_file(str(outpath), title=outpath.stem)
    save(fig)

if SAVE_PLOTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    saved, skipped, failed = [], [], []

    for name in TO_SAVE:
        obj = plot_registry.get(name)
        if obj is None:
            skipped.append((name, "plot object not found (run the source plotting cell first)"))
            continue

        if name == "graph_sizes":
            try:
                out = OUTPUT_DIR / f"{name}.png"
                _save_matplotlib(obj, out)
                saved.append(str(out))
            except Exception as exc:
                failed.append((name, str(exc)))
            continue

        figs = _bokeh_figures(obj)
        for idx, fig in enumerate(figs, start=1):
            suffix = f"_{idx}" if len(figs) > 1 else ""
            out = OUTPUT_DIR / f"{name}{suffix}.html"
            try:
                _save_bokeh_html(fig, out)
                saved.append(str(out))
            except Exception as exc:
                failed.append((f"{name}{suffix}", str(exc)))

    print(f"Saved {len(saved)} plot files")
    if saved:
        print("\n".join(saved))
    if skipped:
        print("\nSkipped:")
        for n, msg in skipped:
            print(f"- {n}: {msg}")
    if failed:
        print("\nFailed:")
        for n, msg in failed:
            print(f"- {n}: {msg}")
else:
    print("Set SAVE_PLOTS = True and rerun this cell to export selected plots.")


INFO:bokeh.io.state:Session output file 'saved_plots/plot_true_graph.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/embedding_training_1.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/embedding_training_2.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/embedding_training_3.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/neighbor_performance_1.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/neighbor_performance_2.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/neighbor_performance_3.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/predicted_graph_1.html' already exists, will be overwritten.
INFO:bokeh.io.state:Session output file 'saved_plots/predicted_gra

Saved 17 plot files
saved_plots/plot_true_graph.html
saved_plots/embedding_training_1.html
saved_plots/embedding_training_2.html
saved_plots/embedding_training_3.html
saved_plots/neighbor_performance_1.html
saved_plots/neighbor_performance_2.html
saved_plots/neighbor_performance_3.html
saved_plots/predicted_graph_1.html
saved_plots/predicted_graph_2.html
saved_plots/track_lengths_1.html
saved_plots/track_lengths_2.html
saved_plots/graph_sizes.png
saved_plots/gnn_training_1.html
saved_plots/gnn_training_2.html
saved_plots/gnn_training_3.html
saved_plots/edge_performance_1.html
saved_plots/edge_performance_2.html
